In [0]:
%pip install xgboost
%restart_python

In [0]:
import importlib
import pandas as pd
import numpy as np
# import src.utils.fetch_lighthouse_data as fld
# importlib.reload(fld)
# from scipy.stats import percentileofscore
import matplotlib.pyplot as plt

# from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, PredictionErrorDisplay

from xgboost import XGBRegressor

# API_KEY = dbutils.secrets.get(
#     scope="site_speed_project",
#     key="google_psi_api_key"
# )

In [0]:
# {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 500, 'subsample': 0.8}

In [0]:
df = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null
''').toPandas()

In [0]:
df.isnull().sum(axis=0)

In [0]:
df_copy = df.copy()

high_missing_col = ['mainthread_garbageCollection','EXPERIMENTAL_TIME_TO_FIRST_BYTE','INTERACTION_TO_NEXT_PAINT']
df_copy = df_copy.drop(columns=high_missing_col)
df_copy=df_copy.dropna()

y = df_copy['largest-contentful-paint']

drop_cols = ['domain', 'url', 'performance_score', 'largest-contentful-paint','cumulative-layout-shift','first-contentful-paint','total-blocking-time','speed-index']
df_copy = df_copy.drop(columns=drop_cols)


In [0]:
X_encoded = pd.get_dummies(
    df_copy,
    columns=["strategy"],
    drop_first=False
)

In [0]:
X_encoded

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [0]:
def run_model(model, X_train, X_test, y_train, y_test, log=False, top_features=None):
    scores = cross_val_score(model, X_train[top_features], y_train, cv=5, scoring="neg_mean_absolute_error")
    print("Scores AVG: ", scores.mean())
    print("Scores STD: ", scores.std())

    model.fit(X_train[top_features], y_train)
    print(model.score(X_train[top_features], y_train))
    y_pred = model.predict(X_test[top_features])
    if log:
        y_pred = np.expm1(y_pred)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    display = PredictionErrorDisplay(y_true=y_test, y_pred=y_pred)
    display.plot()

    print(f"R-squared: {r2}")
    print(f"Root Mean Squared Error: {rmse}")
    print(f"Mean Absolute Error: {mae}")

In [0]:
%skip
model = GradientBoostingRegressor(learning_rate=0.05, max_depth=5, min_samples_leaf=5, n_estimators=500, subsample=0.8, random_state=42)
run_model(model, X_train, X_test, y_train, y_test, top_features=X_encoded.columns)


In [0]:
# {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 500, 'subsample': 0.8}

In [0]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror"
)

scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
display = PredictionErrorDisplay(y_true=y_test, y_pred=y_pred)
display.plot()

print("XGBoost CV scores:", scores)
print("Mean CV R2:", scores.mean())
print("CV std:", scores.std())

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
# Log scale Y
y_train_log = np.log1p(y_train)

In [0]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror",
    eval_metric="mae",
)

scores = cross_val_score(
    xgb,
    X_train,
    y_train_log,
    cv=5,
    scoring="neg_mean_absolute_error"
)

xgb.fit(X_train, y_train_log)
y_pred_log = xgb.predict(X_test)
y_pred = np.expm1(y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
display = PredictionErrorDisplay(y_true=y_test, y_pred=y_pred)
display.plot()

print("XGBoost CV scores:", scores)
print("Mean CV R2:", scores.mean())
print("CV std:", scores.std())

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
clipped_y = y_train.copy()
upper = clipped_y.quantile(0.997)
clipped_y = clipped_y.clip(0, upper)

In [0]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror",
    eval_metric="mae",
)

scores = cross_val_score(
    xgb,
    X_train,
    clipped_y,
    cv=5,
    scoring="neg_mean_absolute_error"
)

xgb.fit(X_train, clipped_y)
y_pred = xgb.predict(X_test)
# y_pred = np.expm1(y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
display = PredictionErrorDisplay(y_true=y_test, y_pred=y_pred)
display.plot()

print("XGBoost CV scores:", scores)
print("Mean CV R2:", scores.mean())
print("CV std:", scores.std())

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
# clip y then log scale

clipped_y = y_train.copy()
# upper = clipped_y.quantile(0.999)
# clipped_y = clipped_y.clip(0, upper)

y_train_log = np.log1p(clipped_y)

In [0]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror",
    eval_metric="mae",
)

scores = cross_val_score(
    xgb,
    X_train,
    y_train_log,
    cv=5,
    scoring="neg_mean_absolute_error"
)

xgb.fit(X_train, y_train_log)
y_pred_log = xgb.predict(X_test)
y_pred = np.expm1(y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
display = PredictionErrorDisplay(y_true=y_test, y_pred=y_pred)
display.plot()

print("XGBoost CV scores:", scores)
print("Mean CV R2:", scores.mean())
print("CV std:", scores.std())

print(f"R-squared: {r2}")
print(f"Root Mean Squared Error: {rmse}")
print(f"Mean Absolute Error: {mae}")

In [0]:
%skip
xgb = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

param_grid = {
    "n_estimators": [300, 500, 700],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [3, 5, 10],
    "reg_lambda": [1, 5, 10]
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train_log)

In [0]:
%skip
best_model = grid.best_estimator_

print("Best parameters:")
print(grid.best_params_)

print("Best CV score:")
print(grid.best_score_)

In [0]:
%skip
best_xgb = XGBRegressor(
    n_estimators=700,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=1.0,
    min_child_weight=5,
    reg_alpha=1,
    reg_lambda=1.0,
    random_state=42,
    objective="reg:squarederror",
    eval_metric="mae",
)

In [0]:

xgb = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

param_grid = {
    "n_estimators": [600, 700, 800],
    "learning_rate": [0.1, 0.2],
    "max_depth": [5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.9, 1.0],
    "min_child_weight": [4, 5, 6],
    "reg_lambda": [0, 0.1, 1, 3, 5, 6],
    "gamma":[0, 0.2]
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=4,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train_log)
best_model = grid.best_estimator_

print("Best parameters:")
print(grid.best_params_)

print("Best CV score:")
print(grid.best_score_)

In [0]:
%skip
{'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 800, 'reg_lambda': 6, 'subsample': 0.8}
Best CV score:
-0.16212932323679158

In [0]:
%skip
grid.best_params_